In [1]:
import time
import tracemalloc
import matplotlib.pyplot as plt
from itertools import permutations

# Standard profiling wrapper for memory and time
def profile_performance(func, *args, **kwargs):
    tracemalloc.start()
    start_time = time.perf_counter()
    
    result = func(*args, **kwargs)
    
    end_time = time.perf_counter()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    return result, end_time - start_time, peak / (1024 * 1024)

print("Environment initialized with performance profiling.")

Environment initialized with performance profiling.


In [2]:
# Task 3a: Define flight and crew data
# Format: (Flight ID, Start Time, End Time)
flights = [('F1', 9, 11), ('F2', 10, 12), ('F3', 12, 14), ('F4', 13, 15), ('F5', 15, 17)]
crew_members = ['C1', 'C2']
min_rest_period = 1 # 1 hour rest required between flights [cite: 191]

def is_feasible(assigned_flights, new_flight):
    """Constraint Checker: No overlap and satisfies rest time [cite: 191]"""
    for f_id, start, end in assigned_flights:
        # Check for time overlap or insufficient rest period
        if not (new_flight[1] >= end + min_rest_period or new_flight[2] <= start - min_rest_period):
            return False
    return True

def solve_scheduling(flight_idx, assignments):
    """Backtracking algorithm to assign flights to crew [cite: 193]"""
    if flight_idx == len(flights):
        return assignments

    flight = flights[flight_idx]
    for crew in crew_members:
        if is_feasible(assignments[crew], flight):
            # Branch: Assign flight
            assignments[crew].append(flight)
            
            # Recurse
            if solve_scheduling(flight_idx + 1, assignments):
                return assignments
            
            # Backtrack [cite: 193]
            assignments[crew].pop()
            
    return None

# Execution
initial_assignments = {crew: [] for crew in crew_members}
schedule_result, exec_time, mem_peak = profile_performance(solve_scheduling, 0, initial_assignments)

print(f"Final Schedule Assignment: {schedule_result}")
print(f"Profiling: {exec_time:.6f}s | Peak Memory: {mem_peak:.6f} MB")

Final Schedule Assignment: {'C1': [('F1', 9, 11), ('F3', 12, 14), ('F5', 15, 17)], 'C2': [('F2', 10, 12), ('F4', 13, 15)]}
Profiling: 0.000300s | Peak Memory: 0.000168 MB


In [3]:
def naive_search(text, pattern):
    """Naive String Matching: O(m*(n-m+1))"""
    n, m = len(text), len(pattern)
    comparisons = 0
    results = []
    for i in range(n - m + 1):
        match = True
        for j in range(m):
            comparisons += 1
            if text[i+j] != pattern[j]:
                match = False
                break
        if match: results.append(i)
    return results, comparisons

def kmp_search(text, pattern):
    """KMP String Matching: O(n + m) [cite: 166]"""
    n, m = len(text), len(pattern)
    comparisons = 0
    
    # Precompute pi table (LPS)
    pi = [0] * m
    j = 0
    for i in range(1, m):
        while j > 0 and pattern[i] != pattern[j]:
            j = pi[j-1]
        if pattern[i] == pattern[j]:
            j += 1
        pi[i] = j
        
    # Search
    results = []
    q = 0 
    for i in range(n):
        comparisons += 1
        while q > 0 and pattern[q] != text[i]:
            q = pi[q-1]
        if pattern[q] == text[i]:
            q += 1
        if q == m:
            results.append(i - m + 1)
            q = pi[q-1]
            
    return results, comparisons

# Comparison Test
test_text = "ABABDABACDABABCABAB" * 100
test_pattern = "ABABCABAB"

res_naive, time_n, _ = profile_performance(naive_search, test_text, test_pattern)
res_kmp, time_k, _ = profile_performance(kmp_search, test_text, test_pattern)

print(f"Naive Comparisons: {res_naive[1]} | Time: {time_n:.6f}s")
print(f"KMP Comparisons: {res_kmp[1]} | Time: {time_k:.6f}s")

Naive Comparisons: 4682 | Time: 0.011600s
KMP Comparisons: 1900 | Time: 0.002900s
